# `IDP-CV`: Invoice data extraction
**With `Docling` processing, `SentenceTransformer` embeddings and `spaCy` NER**

## Dependencies

In [1]:
import logging
import os
import pandas as pd
import sys
import torch
import warnings

from pathlib import Path

from docling.datamodel.settings import settings
from run_extraction import process_and_extract_results

/home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Global Logic / Settings Configuration

In [2]:
# Force offline mode for transformers/sentence-transformers
os.environ['TRANSFORMERS_OFFLINE'] = '1'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('./data/example_invoices/file_inputs/')
OUTPUT_DIR = Path('./data/example_invoices/idp-cv_outputs/')

# docling runtime settings
settings.perf.doc_batch_concurrency = 4
settings.perf.doc_batch_size = 4

# Logger configuration for notebooks
logging.basicConfig(level=logging.INFO, format='%(message)s', stream=sys.stdout, force=True)
logging.getLogger('RapidOCR').setLevel(logging.WARNING)
logging.getLogger('docling').setLevel(logging.WARNING)
logging.getLogger('idp_cv').setLevel(logging.DEBUG)
warnings.filterwarnings('ignore', category=UserWarning, module='docling')

## Schema-aware Tables and Summary extraction
- Processes PDFs and document images with `Docling`'s default `DocumentConverter` for OCR, table and layout detection
- Disambiguates table column labels by applying lexical and semantic ranking with schema aliases using `TableFieldMapper`
- Extracts key/value fields from `DoclingDocument` via label ranking with schema and `spaCy` NER validation `DocumentFieldExtractor` 
- Visualises text and table field boxes on page images (**TODO**: labels)

In [3]:
# Run the unified execution procedure
results = process_and_extract_results(input_dir=DATA_DIR, output_dir=OUTPUT_DIR, device=DEVICE)

Load pretrained SentenceTransformer: ibm-granite/granite-embedding-small-english-r2


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.56it/s]

Found 7 files to process in data/example_invoices/file_inputs
Starting extraction with 4 background workers...



[INFO] 2026-03-25 18:09:07,585 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 18:09:07,619 [RapidOCR] download_file.py:60: File exists and is valid: /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-25 18:09:07,621 [RapidOCR] main.py:53: Using /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-25 18:09:07,719 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 18:09:07,723 [RapidOCR] download_file.py:60: File exists and is valid: /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-25 18:09:07,724 [RapidOCR] main.py:53: Using /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-25 18:09:07,771 [RapidOCR] base.py:22: Using engine_name: onnx

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Mapping table with columns: ['item', 'desc', 'price', 'qty', 'line total']
Using LexicalMapper with aliases: {'desc': ['description', 'desc', 'item description', 'service', 'product', 'details', 'item'], 'qty': ['quantity', 'qty', 'quant', 'units', 'count'], 'price': ['unit price', 'price', 'rate', 'unit cost', 'net price', 'cost per unit', 'cpu'], 'total': ['gross_worth', 'gross total', 'grand total', 'invoice total', 'line_total', 'total_worth', 'total amount', 'total due', 'balance due', 'total balance', 'total', 'balance', 'balance amount', 'full amount'], 'net': ['net worth', 'net amount', 'net total', 'subtotal', 'sub total', 'amount before tax'], 'tax': ['tax', 'vat', 'gst', 'tax amount', 'vat amount', 'gst amount', 'tax due', 'vat due', 'gst due', 'total tax'], 'tax_rate': ['tax (%)', 'vat (%)', 'gst (%)', 'tax %', 'vat %', 'gst %', 'tax rate', 'vat rate', 'gst rate'], 'shipping': ['shipping', 'deliv

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

      Rejected value candidate 'for' - No digit for address and not enough entities in {'LOC', 'FAC', 'GPE'} - for value-type 'address'
      Rejected value candidate 'Thanks' - No digit for address and not enough entities in {'LOC', 'FAC', 'GPE'} - for value-type 'address'
    - Strategy C ngram candidates: ['Terms:']
      Rejected value candidate 'Terms:' - Failed alphanumeric regex - for value-type 'address'
    - Strategy C ngram candidates: ['']
Field 'issuer_tax':
    - Strategy C ngram candidates: ['SuperStore']
    - Strategy C ngram candidates: ['Standard Class', 'Class', 'Standard']
    - Strategy C ngram candidates: ['Ship Mode:', 'Mode:', 'Ship']
    - Strategy C ngram candidates: ['Notes:']
    - Strategy C ngram candidates: ['Thanks for your business!', 'for your business!', 'Thanks for your', 'your business!', 'for your', 'Thanks for', 'business!', 'your', 'for', 'Thanks']
    - Strategy C ngram candidates: ['Terms:']
    - Strategy C ngram candidates: ['']
Field 'issue

Batches: 100%|██████████| 1/1 [00:00<00:00, 24.77it/s]

Field Key Mapping Results:

  - Field "inv_no" matched key "Invoice #:" (score: 1.00 [lexical]) at 1 locations: group[0]: text[3]
    - Candidates considered:
    'Invoice #:' (1.00 [lexical]), 'Invoice' (0.88 [lexical]), 'Invoice -' (0.80 [lexical]), 'Invoice -' (0.78 [lexical]), 'Invoice - Oceanic' (0.53 [lexical]), 'Invoice #: INV-8114' (0.53 [lexical]), 'Invoice #: INV-8114' (0.47 [lexical]), 'INV-8114' (0.38 [lexical]), 'INV-8114' (0.38 [lexical]), 'Oceanic' (0.36 [lexical]), '- Oceanic' (0.36 [lexical]), '#: INV-8114' (0.27 [lexical]), '#: INV-8114' (0.27 [lexical]), 'Marine Drive Miami,' (0.26 [lexical]), 'Jane Smith 300' (0.21 [lexical]), '2025-07-05 PO:' (0.21 [lexical]), '2025-07-05 PO:' (0.21 [lexical]), '-' (0.00 [lexical]), '-' (0.00 [lexical]), '300' (0.00 [lexical]), '300' (0.00 [lexical]), '2025-07-05' (0.00 [lexical]), '2025-07-05' (0.00 [lexical]), '$9470.66' (0.00 [lexical]), '$9470.66' (0.00 [lexical]), '$10158.78' (0.00 [lexical]), '$10158.78' (0.00 [lexical]), 'In


  - Field "shipping" matched key "Shipping:" (score: 1.00 [lexical]) at 1 locations: group[0]: text[4]
    - Candidates considered:
    'Shipping:' (1.00 [lexical]), 'Shipping: $25.17' (0.56 [lexical]), 'Shipping: $25.17' (0.56 [lexical]), '$662.95 Shipping:' (0.53 [lexical]), 'Shipping: $25.17 Total:' (0.48 [lexical]), 'Drive' (0.44 [lexical]), '(7%): $662.95 Shipping:' (0.39 [lexical]), '$662.95 Shipping: $25.17' (0.38 [lexical]), '$662.95 Shipping: $25.17' (0.38 [lexical]), 'Elm St' (0.36 [lexical]), 'Oceanic Corp.' (0.31 [lexical]), 'Elm St' (0.31 [lexical]), '300 Elm St' (0.29 [lexical]), '- Oceanic Corp.' (0.27 [lexical]), 'Miami,' (0.22 [lexical]), 'Elm' (0.22 [lexical]), 'Shipping:' (1.00 [semantic])

  - Field "total" matched key "Total:" (score: 1.00 [lexical]) at 1 locations: group[0]: text[4]
    - Candidates considered:
    'Total:' (1.00 [lexical]), 'Invoice - Oceanic' (0.59 [lexical]), '$25.17 Total:' (0.54 [lexical]), '$25.17 Total:' (0.54 [lexical]), 'To:' (0.50 [lexi

In [4]:
# Visualization and Formatting for Summaries
summary_data = []
for res in results:
    if 'summary' in res:
        entry = res['summary'].copy()
        entry['Document'] = res['Document']
        summary_data.append(entry)

if summary_data:
    df = pd.DataFrame(summary_data)
    # Ensure 'Document' is the first column
    cols = ['Document'] + [c for c in df.columns if c != 'Document']
    df = df[cols].set_index('Document')

    num_cols = df.select_dtypes(include='number').columns
    display(df.style.format({c: '{:.2f}' for c in num_cols}, na_rep='None'))
else:
    print('No results found.')

,invoice_number,order_id,invoice_date,due_date,total_amount,net_amount,tax_amount,tax_rate,shipping_cost,vendor_name,buyer_name,vendor_tax_id,vendor_banking,buyer_tax_id,vendor_address,buyer_address
Document,,,,,,,,,,,,,,,,
inv1,INV-6059,None,2025-07-25,None,9251.14,8621.10,None,None,None,Oceanic Corp.,Acme LLC,None,None,None,"101 Marine Drive Miami, FL 33101","55 Industrial Rd Chicago, IL 60601"
inv2,None,None,None,None,5175.49,4824.45,337.71,0.07,13.33,Skyline Solutions,None,None,None,None,"456 Cloud St Austin, TX 73301",None
inv3,37535203,None,2015-07-13,None,475.75,432.50,43.25,0.10,None,Garcia-Burton,Miranda-Nielsen,916-83-8601,GB30ICA043176037560423,989-84-4547,"0340 Moran Port Pattonborough, Wl 65859","8668 Rangel Stravenue Suite 392 Port Kristen, SD 88510"
inv4,13310164,None,2020-01-16,None,18749.50,17045.00,1704.50,0.10,None,Montgomery LLC,Campbell-Flores,924-83-8802,GB13LKIT26418644296553,999-80-5625,"75081 David Corners Lake Kellyview, NH 03078","34813 Davis Views Jasonhaven, MO 32671"
inv5,1234NA1,PO-65652345,2025-07-08,2025-07-16,36.44,30.37,6.07,0.20,None,John Doe Electronics ltd.,Jane Doe,None,None,None,None,"123, Apple str."
inv6,INV-8114,PO-49113,2025-07-05,None,10158.78,9470.66,662.95,0.07,25.17,Oceanic Corp.,Jane Smith,None,None,None,"101 Marine Drive Miami, FL 33101","300 Elm St Seattle, WA 98101"
inv7,16892,ES-2012-JH15820139-41012,2012-04-13,None,1337.97,1250.04,None,None,87.93,Super Store,John Huston,None,None,None,None,"Crewe, England, United Kingdom"


In [5]:
[item.model_dump(exclude_none=True) for res in results if 'line_items' in res for item in res['line_items'][0]]

[{'description': 'monitor',
  'quantity': 7,
  'unit_price': 346.87,
  'total_amount': 2428.09},
 {'description': 'desk chair',
  'quantity': 10,
  'unit_price': 488.03,
  'total_amount': 4880.3},
 {'description': 'laptop',
  'quantity': 7,
  'unit_price': 187.53,
  'total_amount': 1312.71},
 {'description': 'keyboard', 'quantity': 4, 'total_amount': 436.64},
 {'description': 'desk chair', 'quantity': 7, 'total_amount': 2377.34},
 {'description': 'software license', 'quantity': 5, 'total_amount': 272.15},
 {'description': 'laptop', 'quantity': 4, 'total_amount': 1738.32},
 {'description': 'q&aaday:5-yearjournal',
  'quantity': 3,
  'unit_price': 4.49,
  'total_amount': 14.82,
  'net_amount': 13.47,
  'tax_rate': 10.0},
 {'description': "poorcharlie'salmanackthewit andwisdomofcharlest munger3rded.brandnew",
  'quantity': 5,
  'unit_price': 64.95,
  'total_amount': 357.23,
  'net_amount': 324.75,
  'tax_rate': 10.0},
 {'description': "againstallodds:theuntold storyofcanada'sunlikely hock

In [6]:
[item.model_dump(exclude_none=True) for res in results if 'summary_table' in res for item in res['summary_table'][0]]

[{'total_amount': 475.75,
  'net_amount': 432.5,
  'tax_amount': 43.25,
  'tax_rate': 10.0},
 {'total_amount': 475.75, 'net_amount': 432.5, 'tax_amount': 43.25},
 {'total_amount': 18749.5,
  'net_amount': 17045.0,
  'tax_amount': 1704.5,
  'tax_rate': 10.0},
 {'total_amount': 18749.5, 'net_amount': 17045.0, 'tax_amount': 1704.5},
 {'total_amount': 1337.97, 'net_amount': 1250.04, 'shipping_cost': 87.93}]

# END